# NEAR activity: real users or bots? (reproducibility notebook)

Window **2026-09-08 → 2026-10-07**. Every number in `REPORT.md` comes from the four CSVs in `results/`, which the SQL in `queries/` produced from `bigquery-public-data.crypto_near_mainnet_us`.

This notebook lets you:
1. check the shipped results against their SHA-256 fingerprints (no Google account needed),
2. read the SQL and the data behind each claim,
3. rebuild the charts,
4. optionally re-run every query live on BigQuery (needs credentials mounted at `/secrets/gcp.json` and `BQ_PROJECT` set).

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

PACK = Path(os.environ.get('REPRO_ROOT', '.')).resolve()
RUN = [sys.executable, '/work/repro/run.py']
manifest = json.loads((PACK / 'manifest.json').read_text())
pd.DataFrame(manifest['queries']).T[['window', 'rows', 'bytes_processed', 'result_sha256']]

## 1. Verify the shipped results (offline)

In [ ]:
print(subprocess.run(RUN + ['verify-cached'], capture_output=True, text=True).stdout)

## 2. The SQL and the data behind each claim

In [ ]:
def show(name):
    sql = (PACK / 'queries' / f'{name}.sql').read_text()
    display(Markdown(f'### {name}\n```sql\n{sql}\n```'))
    return pd.read_csv(PACK / 'results' / f'{name}.csv')

buckets = show('q3_signer_buckets')
buckets

In [ ]:
top = show('q4_top_signers')
top.sort_values('tx', ascending=False)

In [ ]:
daily = show('q1_daily_activity').merge(pd.read_csv(PACK / 'results' / 'q2_concentration.csv'), on=['block_date', 'tx_count'])
daily.sort_values('block_date')

## 3. Rebuild the charts

In [ ]:
print(subprocess.run(RUN + ['charts'], capture_output=True, text=True).stdout)
for png in sorted((PACK / 'charts').glob('*.png')):
    display(Image(filename=str(png), width=900))

## 4. Optional: re-run every query live on BigQuery

Costs about 3 GB of query volume, billed to `BQ_PROJECT`. Each query is dry-run first and blocked above `MAX_GB_PER_QUERY`. Set `LIVE = True` to run it.

In [ ]:
LIVE = False
if LIVE:
    r = subprocess.run(RUN + ['verify'], capture_output=True, text=True)
    print(r.stdout, r.stderr[-2000:])
else:
    print('Skipped. Set LIVE = True (needs /secrets/gcp.json and BQ_PROJECT).')